# Derivatives and Gradients

Companion notebook for the [Derivatives and Gradients](https://ml-viz.vercel.app/courses/calculus-for-ml/01-derivatives-and-gradients) lesson.

We'll visualize derivatives, compute gradients analytically and numerically, and implement gradient descent.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Derivative as slope of the tangent line

In [ ]:
def f(x): return x**3 - 2*x**2 + x
def df(x): return 3*x**2 - 4*x + 1  # analytical derivative

x = np.linspace(-0.5, 2.5, 400)
tangent_points = [0.2, 1.0, 2.0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, f(x), color='#6366f1', lw=2.5, label='f(x) = x³ - 2x² + x')

colors = ['#f97316', '#2dd4bf', '#f59e0b']
for x0, color in zip(tangent_points, colors):
    slope = df(x0)
    tangent = f(x0) + slope * (x - x0)
    # Only draw tangent in a window
    mask = abs(x - x0) < 0.4
    ax.plot(x[mask], tangent[mask], '--', color=color, lw=2,
            label=f"f'({x0}) = {slope:.2f}")
    ax.scatter([x0], [f(x0)], color=color, s=60, zorder=5)

ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.grid(True, alpha=0.2); ax.legend()
ax.set_title('Derivative = slope of tangent line', pad=12)
plt.tight_layout(); plt.show()

## Numerical vs analytical gradient verification

In [ ]:
def g(x):
    """A function of multiple variables."""
    return x[0]**2 + 2*x[1]**2 + x[0]*x[1]

def grad_g(x):
    """Analytical gradient."""
    return np.array([2*x[0] + x[1], 4*x[1] + x[0]])

def numerical_grad(f, x, eps=1e-5):
    """Central difference approximation."""
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += eps
        x_minus = x.copy(); x_minus[i] -= eps
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * eps)
    return grad

test_points = [np.array([1.0, 2.0]), np.array([-1.0, 0.5]), np.array([3.0, -1.0])]

print("{:20s}  {:30s}  {:30s}  {}".format("x", "Analytical", "Numerical", "Max error"))
print("-" * 90)
for x0 in test_points:
    ag = grad_g(x0)
    ng = numerical_grad(g, x0)
    err = np.max(np.abs(ag - ng))
    print("{:20s}  {:30s}  {:30s}  {:.2e}".format(
        str(x0), str(ag.round(4)), str(ng.round(4)), err))

## Gradient descent on a 2D function

In [ ]:
def loss(w): return (w[0] - 3)**2 + 2*(w[1] + 1)**2   # minimum at (3, -1)
def grad_loss(w): return np.array([2*(w[0]-3), 4*(w[1]+1)])

# Run gradient descent
w = np.array([-2.0, 3.0])   # start far from minimum
lr = 0.1
path = [w.copy()]

for _ in range(30):
    w = w - lr * grad_loss(w)
    path.append(w.copy())

path = np.array(path)

# Plot
xx, yy = np.meshgrid(np.linspace(-3, 5, 200), np.linspace(-3, 5, 200))
Z = (xx - 3)**2 + 2*(yy + 1)**2

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(xx, yy, Z, levels=20, cmap='twilight', alpha=0.7)
ax.contour(xx, yy, Z, levels=20, colors='white', alpha=0.2, linewidths=0.5)
plt.colorbar(cs, ax=ax, label='Loss')

ax.plot(path[:, 0], path[:, 1], 'o-', color='#f97316', ms=5, lw=2, label='GD path')
ax.scatter([path[0,0]], [path[0,1]], color='#2dd4bf', s=100, zorder=6, label='Start')
ax.scatter([3], [-1], marker='*', color='#f59e0b', s=200, zorder=6, label='Minimum')

ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Gradient descent on f(w₁,w₂) = (w₁−3)² + 2(w₂+1)²', pad=12)
ax.legend()
plt.tight_layout(); plt.show()

print(f'Final w: {path[-1].round(4)}  (true minimum: [3, -1])')
print(f'Final loss: {loss(path[-1]):.6f}')

## The sigmoid derivative, derived and verified

Using the chain rule on $\sigma(x) = (1 + e^{-x})^{-1}$:

$$\sigma'(x) = \frac{e^{-x}}{(1+e^{-x})^2} = \frac{1}{1+e^{-x}}\cdot\frac{e^{-x}}{1+e^{-x}} = \sigma(x)\,(1-\sigma(x)).$$

So the gradient is a cheap function of the forward output. Below we confirm the closed form matches a finite-difference derivative, and see why a saturated unit ($|x|$ large) has a near-zero gradient.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_deriv_closed(x):
    s = sigmoid(x)
    return s * (1.0 - s)            # the σ(1-σ) shortcut

# Confirm the closed form matches a finite-difference derivative everywhere.
xs = np.linspace(-6, 6, 25)
eps = 1e-6
numeric = (sigmoid(xs + eps) - sigmoid(xs - eps)) / (2 * eps)
closed  = sigmoid_deriv_closed(xs)
print('max |closed-form - numerical| =', np.max(np.abs(closed - numeric)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, sigmoid(xs), color='#6366f1', lw=2.5, label='σ(x)')
ax.plot(xs, closed, color='#f97316', lw=2.5, label="σ'(x) = σ(1-σ)")
ax.scatter(xs, numeric, color='#2dd4bf', s=12, zorder=5, label='σ′ numerical')
ax.axhline(0, color='#30344a')
ax.set_title("Sigmoid derivative peaks at x=0 (0.25) and vanishes in the tails", pad=10)
ax.set_xlabel('x'); ax.grid(True, alpha=0.2); ax.legend()
plt.tight_layout(); plt.show()
print(f"σ'(0) = {sigmoid_deriv_closed(np.array([0.0]))[0]:.3f}  (max slope)")
print(f"σ'(6) = {sigmoid_deriv_closed(np.array([6.0]))[0]:.5f}  (saturated -> gradient ~0)")